# 🏠 Real Estate Intelligence Platform
## Notebook 01 — Data Inspection & Cleaning

**Project:** AI-Powered Real Estate Intelligence, Valuation & Investment Recommendation Platform  
**Phase:** Phase 2 (Data Collection & Validation) + Phase 3 (Data Cleaning)  
**Dataset:** India House Price Prediction — 2,50,000 records × 23 columns  
**Source:** Kaggle (ankushpanday1/india-house-price-prediction)  

---

### Objectives
- Inspect dataset structure, dtypes, and missing values
- Identify and fix data quality issues
- Detect and handle outliers
- Save cleaned dataset for feature engineering

---

### Issues Found & Fixed
| # | Issue | Action Taken |
|---|---|---|
| 1 | Price_per_SqFt incorrectly calculated in source | Recalculated as (Price_in_Lakhs × 100000) / Size_in_SqFt |
| 2 | Floor_No > Total_Floors in 116,304 rows (46%) | Swapped Floor_No and Total_Floors where condition was true |
| 3 | Price_per_SqFt outliers — 19,723 rows (7.9%) above IQR upper bound | Capped at 95th percentile (₹40,568/sqft) |
| 4 | ID column — no modeling value | Dropped |

In [1]:
# Imports 
import numpy as np 
import pandas as pd

In [2]:
df = pd.read_csv('../data/raw/india_housing_prices.csv')
df.head()

,ID,State,City,Locality,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,...,Age_of_Property,Nearby_Schools,Nearby_Hospitals,Public_Transport_Accessibility,Parking_Space,Security,Amenities,Facing,Owner_Type,Availability_Status
0,1,Tamil Nadu,Chennai,Locality_84,Apartment,1,4740,489.76,0.10,1990,...,35,10,3,High,No,No,"Playground, Gym, Garden, Pool, Clubhouse",West,Owner,Ready_to_Move
1,2,Maharashtra,Pune,Locality_490,Independent House,3,2364,195.52,0.08,2008,...,17,8,1,Low,No,Yes,"Playground, Clubhouse, Pool, Gym, Garden",North,Builder,Under_Construction
2,3,Punjab,Ludhiana,Locality_167,Apartment,2,3642,183.79,0.05,1997,...,28,9,8,Low,Yes,No,"Clubhouse, Pool, Playground, Gym",South,Broker,Ready_to_Move
3,4,Rajasthan,Jodhpur,Locality_393,Independent House,2,2741,300.29,0.11,1991,...,34,5,7,High,Yes,Yes,"Playground, Clubhouse, Gym, Pool, Garden",North,Builder,Ready_to_Move
4,5,Rajasthan,Jaipur,Locality_466,Villa,4,4823,182.90,0.04,2002,...,23,4,9,Low,No,Yes,"Playground, Garden, Gym, Pool, Clubhouse",East,Builder,Ready_to_Move


In [3]:
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())
print("\nSample:\n", df.head(3))

Shape: (250000, 23)

Dtypes:
 ID                                  int64
State                              object
City                               object
Locality                           object
Property_Type                      object
BHK                                 int64
Size_in_SqFt                        int64
Price_in_Lakhs                    float64
Price_per_SqFt                    float64
Year_Built                          int64
Furnished_Status                   object
Floor_No                            int64
Total_Floors                        int64
Age_of_Property                     int64
Nearby_Schools                      int64
Nearby_Hospitals                    int64
Public_Transport_Accessibility     object
Parking_Space                      object
Security                           object
Amenities                          object
Facing                             object
Owner_Type                         object
Availability_Status                object
dtyp

In [4]:
# Check 1 — price and size ranges (catch zeros/negatives/absurd values)
print(df[['Price_in_Lakhs', 'Size_in_SqFt', 'BHK', 
          'Floor_No', 'Total_Floors', 'Age_of_Property']].describe())

# Check 2 — all categorical unique values
cats = ['Property_Type', 'Furnished_Status', 'Public_Transport_Accessibility',
        'Parking_Space', 'Security', 'Facing', 'Owner_Type', 
        'Availability_Status', 'Amenities']
for col in cats:
    print(f"\n{col} ({df[col].nunique()} unique):\n", df[col].value_counts())

       Price_in_Lakhs   Size_in_SqFt            BHK       Floor_No  \
count   250000.000000  250000.000000  250000.000000  250000.000000   
mean       254.586854    2749.813216       2.999396      14.966800   
std        141.349921    1300.606954       1.415521       8.948047   
min         10.000000     500.000000       1.000000       0.000000   
25%        132.550000    1623.000000       2.000000       7.000000   
50%        253.870000    2747.000000       3.000000      15.000000   
75%        376.880000    3874.000000       4.000000      23.000000   
max        500.000000    5000.000000       5.000000      30.000000   

        Total_Floors  Age_of_Property  
count  250000.000000    250000.000000  
mean       15.503004        18.479988  
std         8.671618         9.808575  
min         1.000000         2.000000  
25%         8.000000        10.000000  
50%        15.000000        18.000000  
75%        23.000000        27.000000  
max        30.000000        35.000000  

Property

In [5]:
# Check floor logic error
print("Floor > Total Floors:", (df['Floor_No'] > df['Total_Floors']).sum())

# Check Price_per_SqFt issue we spotted
df['Price_per_SqFt_Check'] = (df['Price_in_Lakhs'] * 100000) / df['Size_in_SqFt']
print("\nOriginal Price_per_SqFt sample:", df['Price_per_SqFt'].head(3).values)
print("Recalculated sample:", df['Price_per_SqFt_Check'].head(3).values)

Floor > Total Floors: 116304

Original Price_per_SqFt sample: [0.1  0.08 0.05]
Recalculated sample: [10332.48945148  8270.72758037  5046.40307523]


In [6]:
# FIX 1 — Recalculate Price_per_SqFt correctly
df['Price_per_SqFt'] = ((df['Price_in_Lakhs'] * 100000) / df['Size_in_SqFt']).round(2)

# FIX 2 — Fix Floor_No > Total_Floors
# Where floor exceeds total, swap them
mask = df['Floor_No'] > df['Total_Floors']
df.loc[mask, ['Floor_No', 'Total_Floors']] = df.loc[mask, ['Total_Floors', 'Floor_No']].values

# Verify both fixes
print("Floor > Total Floors remaining:", (df['Floor_No'] > df['Total_Floors']).sum())
print("\nPrice_per_SqFt sample (fixed):", df['Price_per_SqFt'].head(3).values)
print("\nPrice_per_SqFt stats:")
print(df['Price_per_SqFt'].describe())

Floor > Total Floors remaining: 0

Price_per_SqFt sample (fixed): [10332.49  8270.73  5046.4 ]

Price_per_SqFt stats:
count    250000.000000
mean      13058.281773
std       13071.850479
min         202.250000
25%        4802.840000
50%        9244.750000
75%       15987.392500
max       99182.000000
Name: Price_per_SqFt, dtype: float64


In [7]:
# Check outliers using IQR on the 3 most important numeric columns
for col in ['Price_in_Lakhs', 'Size_in_SqFt', 'Price_per_SqFt']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    print(f"{col}: lower={lower:.1f}, upper={upper:.1f}, outliers={outliers} ({outliers/len(df)*100:.1f}%)")

Price_in_Lakhs: lower=-233.9, upper=743.4, outliers=0 (0.0%)
Size_in_SqFt: lower=-1753.5, upper=7250.5, outliers=0 (0.0%)
Price_per_SqFt: lower=-11974.0, upper=32764.2, outliers=19723 (7.9%)


In [8]:
# Cap Price_per_SqFt at upper bound (not remove — data is valid)
upper_cap = df['Price_per_SqFt'].quantile(0.95)
df['Price_per_SqFt'] = df['Price_per_SqFt'].clip(upper=upper_cap)
print("Price_per_SqFt after capping:", df['Price_per_SqFt'].describe())
print("Shape still:", df.shape)

Price_per_SqFt after capping: count    250000.000000
mean      12308.212214
std       10490.249652
min         202.250000
25%        4802.840000
50%        9244.750000
75%       15987.392500
max       40568.997500
Name: Price_per_SqFt, dtype: float64
Shape still: (250000, 24)


In [9]:
df.drop(columns=['ID', 'Year_Built', 'Locality', 'Price_per_SqFt_Check'], inplace=True)
df.to_csv('../data/processed/cleaned_df.csv', index=False)
print("Cleaned dataset saved!")
print("Final shape:", df.shape)
print("Columns:", df.columns.tolist())

Cleaned dataset saved!
Final shape: (250000, 20)
Columns: ['State', 'City', 'Property_Type', 'BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Furnished_Status', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Amenities', 'Facing', 'Owner_Type', 'Availability_Status']


---

## ✅ Notebook 01 Complete

### Summary
| Item | Value |
|---|---|
| Original shape | (2,50,000 × 23) |
| Final shape | (2,50,000 × 20) |
| Missing values | 0 |
| Duplicate rows | 0 |
| Outliers handled | 19,723 rows capped (Price_per_SqFt) |
| Logic errors fixed | 1,16,304 rows (Floor_No > Total_Floors swapped) |
| Columns dropped | ID, Year_Built, Locality, Price_per_SqFt_Check |
| Reason for drops | ID (no modeling value) · Year_Built (redundant, Age_of_Property exists) · Locality (generic codes, City captures location) · Price_per_SqFt_Check (verification column only) |
| Output file | data/cleaned_df.csv |

### Remaining 20 Columns
| # | Column | Type |
|---|---|---|
| 1 | State | Categorical |
| 2 | City | Categorical |
| 3 | Property_Type | Categorical |
| 4 | BHK | Numerical |
| 5 | Size_in_SqFt | Numerical |
| 6 | Price_in_Lakhs | Numerical (Target) |
| 7 | Price_per_SqFt | Numerical |
| 8 | Furnished_Status | Categorical |
| 9 | Floor_No | Numerical |
| 10 | Total_Floors | Numerical |
| 11 | Age_of_Property | Numerical |
| 12 | Nearby_Schools | Numerical |
| 13 | Nearby_Hospitals | Numerical |
| 14 | Public_Transport_Accessibility | Categorical |
| 15 | Parking_Space | Categorical |
| 16 | Security | Categorical |
| 17 | Amenities | Categorical |
| 18 | Facing | Categorical |
| 19 | Owner_Type | Categorical |
| 20 | Availability_Status | Categorical |

### Next Step
→ Open `02_feature_engineering.ipynb` to engineer new features and encode/scale for modeling